# Module 4 — Add AgentCore Observability

Your agent is **built** (Module 1) and **deployed** (Module 2). The last rung makes it **observable**:
you'll *see* exactly what every request does — which tools it called, how many tokens it used, how long
each step took, and where it failed — in the **Amazon CloudWatch GenAI Observability** dashboard.

### What it takes (no agent code changes)

Observability adds **zero agent code** — the agent here is byte-identical to Module 2. It's three
operational switches plus a look at the dashboard:

| Step | What happens | Where |
|------|--------------|-------|
| **1. Transaction Search** | Account-level switch — makes spans searchable in `/aws/spans` | one-time, account (a script) |
| **2. Deploy** | The agent's image already runs under `opentelemetry-instrument` (ADOT), so it **emits** OTEL spans | `agentcore deploy` |
| **3. Runtime Tracing toggle** | Per-runtime switch — **delivers** the agent's spans to CloudWatch | console, per deployed agent |
| **4. Invoke + view** | Generate traffic (with a session id) and read the trace waterfall | console dashboard |

> Two distinct things have to be true: the agent must **emit** spans (Step 2 — the container's
> `opentelemetry-instrument` wrapper) *and* the runtime must be told to **deliver** them (Step 3 — the
> Tracing toggle). Account-level Transaction Search (Step 1) makes them searchable.

## Why observability?

When an agent runs autonomously through a multi-step loop, "the answer looks wrong" tells you almost
nothing. Observability turns the black box into a glass box:

- **Trace waterfall** — every step and tool call, for debugging
- **Token & cost** — what each invocation costs
- **Session correlation** — group everything by user/session for support
- **Latency & errors** — find the slow step, see where it failed

A production agent you can't see into is one you can't trust or operate.

## What you'll see in a trace (reading, not writing)

The runtime emits spans following **OpenTelemetry GenAI semantic conventions**, which is what lets the
CloudWatch dashboard render them as an agent trace:

```
invoke_agent cos                         ← top-level span for one request
├── gen_ai.operation.name = "invoke_agent"
├── gen_ai.usage.input_tokens / output_tokens
├── session.id = "<your session id>"     ← groups invocations in the same conversation
└── execute_tool <name>                  ← one child span per tool the agent used
    ├── Bash  (ran a script)
    ├── Read  (read financial_data / CLAUDE.md)
    └── Task  (delegated to a subagent)
```

You **read** these conventions to interpret a trace. You do **not** hand-write spans — the AgentCore
runtime does the instrumentation for you. (That's the modern, recommended path; older examples that
hand-built GenAI spans are no longer necessary for runtime-hosted agents.)

## Setup

Complete the one-time environment setup in [`README.md`](./README.md) (`uv sync`, copy `.env`, set
`agentcore/aws-targets.json` to your account/region). Then run the cell below.

In [ ]:
import os, subprocess, json
from dotenv import load_dotenv

load_dotenv()
REGION = os.getenv("AWS_REGION", "us-west-2")
print(f"AWS region: {REGION}")

# Confirm AWS identity (observability is all account-side)
import boto3
try:
    who = boto3.client("sts").get_caller_identity()
    print(f"✅ AWS identity: {who['Arn']}")
except Exception as e:
    print(f"⚠️  AWS credentials not usable: {e}")

## Step 1 — Enable CloudWatch Transaction Search (one-time, account-level)

This is the **only** genuinely new setup in Module 4. Transaction Search is what makes the agent's
OpenTelemetry spans searchable in CloudWatch (they land in the `/aws/spans` log group). It's an
account-level switch — you do it once, not per deployment.

The helper is **idempotent** (safe to re-run): it checks the current state and only changes what's
missing.

In [ ]:
result = subprocess.run(
    ["python", "scripts/enable_transaction_search.py", "--region", REGION],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)
# Note: after first enabling, allow ~10 minutes before spans are fully searchable.

## Step 2 — Deploy the (already-observable) agent

This is the *same* deploy as Module 2 — nothing extra. Because `enableOtel: true` and ADOT are already in
the config/image, the deployed agent is instrumented automatically. Look at the runtime config:

In [ ]:
cfg = json.load(open("agentcore/agentcore.json"))
rt = cfg["runtimes"][0]
print(json.dumps({
    "name": rt["name"], "build": rt["build"], "protocol": rt["protocol"],
    "instrumentation": rt.get("instrumentation"),
}, indent=2))
# enableOtel: true  → the AgentCore runtime wraps the agent with opentelemetry-instrument on deploy.

Deploy from a **terminal** (builds the image in the cloud via CodeBuild, ~several minutes):

```bash
agentcore deploy -y
agentcore status        # confirm the runtime is READY — note its agent id / ARN for the next step
```

The container's `CMD` runs the agent under **`opentelemetry-instrument`** (from `aws-opentelemetry-distro`),
so once deployed the agent **emits** OTEL spans automatically. Next we tell the runtime to **deliver** them.

## Step 3 — Enable Tracing on the runtime (one toggle, in the console)

A freshly deployed AgentCore runtime **emits** spans but doesn't **deliver** them to CloudWatch until you
turn on its **Tracing** toggle. This is a per-runtime switch and is done in the console:

1. Open the **AgentCore → Agent Runtime** page:
   `https://{REGION}.console.aws.amazon.com/bedrock-agentcore/agents` (replace `{REGION}`)
2. Select your agent (the one you just deployed — name starts with `cos`).
3. In the **Tracing** pane, choose **Edit**, toggle **Enable**, and **Save**.

> Once enabled, the agent's spans flow to the `aws/spans` log group and show up in the GenAI
> Observability dashboard. You only do this once per runtime.

*(Why a manual step? For AgentCore **runtime** resources, enabling trace delivery is a console action —
there isn't a public CLI/`agentcore.json` field for it yet. The account-level Transaction Search and the
container's OTEL emission are both automated above; this toggle is the one click that isn't.)*

## Step 4 — Generate traffic (with a session id)

Invoke the deployed agent a couple of times, passing a **session id**. The session id is what groups
related invocations together in the dashboard's *Sessions* view.

```bash
agentcore invoke --session-id "m4-demo-001" "What is our current runway and cash position?"
agentcore invoke --session-id "m4-demo-001" "If we hire 10 engineers, how does that change?"
```

In [ ]:
# You can also invoke from the notebook:
SESSION_ID = "m4-demo-001"
res = subprocess.run(
    ["agentcore", "invoke", "--session-id", SESSION_ID, "What is our current runway?"],
    capture_output=True, text=True,
)
print((res.stdout or res.stderr)[-1500:])

## Step 5 — View the traces

**In the console (the main event):** open the GenAI Observability dashboard — it has **Agents**,
**Sessions**, and **Traces** views. Pick your agent, drill into a session, and open a trace to see the
span waterfall (tool calls, token usage, latency).

```
https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability
```

(Replace `{REGION}`. Allow ~2–10 minutes after invoking for spans to be indexed.)

**Programmatic peek (so this notebook can verify, not just point at the console):** spans land in the
`/aws/spans` CloudWatch log group. We can query it for spans carrying our session id.

In [ ]:
import time

print(f"GenAI dashboard: "
      f"https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability")

# Peek at /aws/spans for spans from our session (best-effort; indexing can lag a few minutes).
logs = boto3.client("logs", region_name=REGION)

def find_spans(session_id, attempts=10, delay=30):
    query = (
        "fields @timestamp, attributes.session.id, name, attributes.gen_ai.operation.name "
        f"| filter attributes.session.id = '{session_id}' "
        "| sort @timestamp desc | limit 20"
    )
    for i in range(attempts):
        start = logs.start_query(
            logGroupName="aws/spans",
            startTime=int(time.time()) - 3600,
            endTime=int(time.time()),
            queryString=query,
        )["queryId"]
        # poll this query
        while True:
            r = logs.get_query_results(queryId=start)
            if r["status"] in ("Complete", "Failed", "Cancelled"):
                break
            time.sleep(2)
        rows = r.get("results", [])
        if rows:
            print(f"✅ Found {len(rows)} span(s) for session '{session_id}':")
            for row in rows[:5]:
                d = {f["field"]: f["value"] for f in row}
                print("  •", d.get("name"), "|", d.get("attributes.gen_ai.operation.name", ""))
            return True
        print(f"  …no spans yet (attempt {i+1}/{attempts}); waiting {delay}s for indexing")
        time.sleep(delay)
    print("⚠️  No spans found yet — check the console dashboard; indexing can take up to ~10 min.")
    return False

# Uncomment to poll (can take several minutes):
# find_spans(SESSION_ID)

## Step 6 — Cleanup

Tear down the runtime to avoid ongoing charges (same as Module 2). Run in a **terminal**:

```bash
agentcore remove agent --name cos
agentcore deploy -y          # applies the removal → destroys the runtime/stack
```

> **Transaction Search stays enabled** — it's an account-level, one-time setting, not per-deployment, and
> there's no charge for leaving it on at low/no indexing sampling.

## Key takeaways

- AgentCore Runtime **auto-instruments** your agent with OpenTelemetry — observability needs **no agent
  code change** (it was already on from Module 2's `enableOtel` + ADOT).
- The one new step is **enabling CloudWatch Transaction Search** (account-level, one-time).
- Traces follow **GenAI semantic conventions** (`gen_ai.*`, `session.id`), which is why the **GenAI
  Observability dashboard** can render the agent's trace waterfall, tokens, and tool calls.
- Pass a **session id** on invoke to correlate a conversation in the dashboard.

🎉 You've climbed the whole ladder: **built → deployed → observable.** (Memory — Module 3 — is the
remaining rung when you're ready.)